# 06 · Corrección del score de selección adversa

**Qué pregunta responde.** ¿Qué cambia al corregir la orientación del score de selección adversa — y sobrevive alguna conclusión apoyada en la versión equivocada? Se había usado `P(adverse)` alto como criterio de selección, lo que equivale a priorizar situaciones peligrosas, no sanas.

**Qué entradas usa.** Solo CSV públicos: `results/key_results.csv` (filas `methodology` y `oof` relacionadas con `nested`/`fusion`). Sin corpus privado y sin aleatoriedad: cuaderno de lectura re-ejecutable.

**Qué produce.** La declaración trazable del error, la tabla de interpretación corregida (`safe_adverse_score = 1 − P(adverse)`) y la evidencia del registro público de que ninguna configuración robusta sobrevive tras corregir.

**Conclusión (2 frases).** La antigua candidata basada en `P(adverse)` queda invalidada: tras corregir la orientación, cero configuraciones `safe_adverse` robustas pasan el protocolo. El error no se borra, se declara — no basta con que una métrica sea positiva; hay que auditar qué significa económicamente.

**Fila de `results/key_results.csv`.** `methodology,nested_v3_corrected,safe_adverse_robust_configs = 0` (NO_GO, «earlier adverse candidate invalidated») — la fila que registra esta invalidación.

In [1]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / 'results').exists():
    ROOT = ROOT.parent

key = pd.read_csv(ROOT / 'results' / 'key_results.csv')
summary = pd.read_csv(ROOT / 'results' / 'final_candidate_summary.csv')
ledger = pd.read_csv(ROOT / 'results' / 'final_candidate_actions_anonymized.csv')

## Qué se corrigió

El score operativo correcto es `safe_adverse_score = 1 - P(adverse)`. Tras corregirlo, la antigua candidata basada en `P(adverse)` deja de ser evidencia válida.

In [2]:
pd.DataFrame([
    {"elemento": "Target adverso", "interpretación correcta": "1 = peor cuartil de ejecución / mayor riesgo"},
    {"elemento": "Score raw P(adverse)", "interpretación correcta": "Alto es peor, no mejor"},
    {"elemento": "Score safe", "interpretación correcta": "1 - P(adverse), alto es más sano"},
    {"elemento": "Consecuencia", "interpretación correcta": "Las políticas raw adversas quedan invalidadas"},
])

,elemento,interpretación correcta
0,Target adverso,1 = peor cuartil de ejecución / mayor riesgo
1,Score raw P(adverse),"Alto es peor, no mejor"
2,Score safe,"1 - P(adverse), alto es más sano"
3,Consecuencia,Las políticas raw adversas quedan invalidadas


## Evidencia resumida en el registro público

El registro de resultados conserva la decisión corregida: no hay configuraciones robustas basadas en `safe_adverse` que pasen el protocolo.

In [3]:
mask = key["phase"].isin(["methodology", "oof"]) | key["experiment"].str.contains("nested|fusion", case=False, na=False)
key.loc[mask, ["phase", "experiment", "metric", "value", "unit", "status", "interpretation"]]

,phase,experiment,metric,value,unit,status,interpretation
8,methodology,nested_v3_corrected,safe_adverse_robust_configs,0,count,NO_GO,Earlier adverse candidate invalidated
9,oof,clean_h60_upstream,test_auc,0.5656,auc,PARTIAL_SIGNAL,Modest causal ranking signal
10,oof,clean_h60_upstream,test_top35_cost0p50_mean,0.1778,proxy_ticks,PARTIAL_SIGNAL,Positive under primary proxy cost
11,oof,clean_h60_upstream,test_top35_cost1p00_mean,-0.1899,proxy_ticks,NO_GO,Does not survive conservative cost
12,oof,clean_fusion_stage2,robust_configs,0,count,NO_GO,Second stage does not create stable policy


## Lectura

Esta corrección refuerza el mensaje metodológico de la memoria: no basta con que una métrica sea positiva; hay que auditar qué significa económicamente. Después de corregir la orientación, la pista no sobrevive como política seleccionable.